In [2]:
# ---------- 0. 샘플 추출 ----------
# 전체(수천 개)를 다 하면 시간이 너무 오래 걸려서, mp 데이터셋에서 20개만 랜덤 샘플링해서 파이프라인을 검증
import pandas as pd
import numpy as np

mp = pd.read_csv("./data/processed/crc_mp_c.csv")
sample = mp.sample(n=20, random_state=42)[["SMILES", "InChIKey"]].reset_index(drop=True)
print(sample.shape)
sample.head()

(20, 2)


,SMILES,InChIKey
0,O=c1[nH]cc(F)c(=O)[nH]1,GHASVSINZRGABV-UHFFFAOYSA-N
1,Oc1c2ccccc2c(O)c2ccccc12,PCFMUWBCZZUMRX-UHFFFAOYSA-N
2,Oc1ccc(-c2ccc(O)cc2)cc1,VCCBEIPGXKNHFW-UHFFFAOYSA-N
3,CCCCCCCCCCCCCCCCCCCCCC,HOWGUJZVBDQJKV-UHFFFAOYSA-N
4,CCC(CC)(CO)CO,XRVCFZPJAHWYTB-UHFFFAOYSA-N


In [3]:
# ---------- 1. PubChem: InChIKey -> CAS 번호 ----------
import requests
import re
import time

CAS_PATTERN = re.compile(r"^\d{2,7}-\d{2}-\d$")

def get_cas_from_inchikey(inchikey):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/synonyms/JSON"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        synonyms = resp.json()["InformationList"]["Information"][0]["Synonym"]
        cas_candidates = [s for s in synonyms if CAS_PATTERN.match(s)]
        if not cas_candidates:
            return None
        # CAS가 여러 개 붙어있는 경우, 가장 짧은(=대표적인) 걸 우선 채택
        return sorted(cas_candidates, key=len)[0]
    except Exception as e:
        print("  에러:", e)
        return None


cas_list = []
for i, inchikey in enumerate(sample["InChIKey"]):
    cas = get_cas_from_inchikey(inchikey)
    cas_list.append(cas)
    print(f"{i+1}/{len(sample)}: {inchikey} -> {cas}")
    time.sleep(0.3)  # PubChem 요청 제한(초당 5회) 고려

sample["CAS"] = cas_list
print(f"\nCAS 확보: {sample['CAS'].notna().sum()} / {len(sample)}")

1/20: GHASVSINZRGABV-UHFFFAOYSA-N -> 51-21-8
2/20: PCFMUWBCZZUMRX-UHFFFAOYSA-N -> 4981-66-2
3/20: VCCBEIPGXKNHFW-UHFFFAOYSA-N -> 92-88-6
4/20: HOWGUJZVBDQJKV-UHFFFAOYSA-N -> 629-97-0
5/20: XRVCFZPJAHWYTB-UHFFFAOYSA-N -> 115-76-4
6/20: DVECBJCOGJRVPX-UHFFFAOYSA-N -> 141-75-3
7/20: FSQQTNAZHBEJLS-UHFFFAOYSA-N -> None
8/20: ZQEIXNIJLIKNTD-UHFFFAOYSA-N -> 57837-19-1
9/20: DPVIABCMTHHTGB-UHFFFAOYSA-N -> 3764-01-0
10/20: DXYYSGDWQCSKKO-UHFFFAOYSA-N -> 120-75-2
11/20: DFWFIQKMSFGDCQ-UHFFFAOYSA-N -> 6190-65-4
12/20: KMHZPJNVPCAUMN-UHFFFAOYSA-N -> 136-25-4
13/20: ANGDWNBGPBMQHW-UHFFFAOYSA-N -> 105-34-0
14/20: FMWLUWPQPKEARP-UHFFFAOYSA-N -> 75-27-4
15/20: OMWSZDODENFLSV-UHFFFAOYSA-N -> 85-19-8
16/20: DFJYZCUIKPGCSG-UHFFFAOYSA-N -> 1871-96-1
17/20: OYHQOLUKZRVURQ-UHFFFAOYSA-N -> 2197-37-7
18/20: XWKFPIODWVPXLX-UHFFFAOYSA-N -> 589-93-5
19/20: GCMNJUJAKQGROZ-UHFFFAOYSA-N -> 580-22-3
20/20: KCDXJAYRVLXPFO-UHFFFAOYSA-N -> 134-96-3

CAS 확보: 19 / 20


In [9]:
# ---------- 2. LookChem: CAS -> 가격 (Selenium 버전) ----------
# pip install selenium  (설치 안 돼있으면 먼저 실행)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import random

options = Options()
driver = webdriver.Chrome(options=options)  # 브라우저 하나만 띄워서 재사용


def get_lookchem_prices_selenium(cas):
    if pd.isna(cas):  # cas is None 대신 pd.isna() 사용 (pandas는 결측값을 NaN으로 저장함)
        return []
    folder = cas.split("-")[0]
    url = f"https://www.lookchem.com/cas-{folder}/{cas}.html"

    try:
        driver.get(url)
        time.sleep(2)  # 페이지 로딩 대기
    except Exception as e:
        print("  요청 에러:", e)
        return []

    page_source = driver.page_source
    if "FOB Price" not in page_source:
        print("  가격 정보 없음 (차단되었거나 다른 페이지)")
        return []

    prices = []
    for m in re.finditer(r"FOB Price:.*?USD\s*\$\s*([\d.]+)-([\d.]+)", page_source):
        low, high = float(m.group(1)), float(m.group(2))
        if low > 0 or high > 0:  # $0.0-0.0 (가격 미기재)는 제외
            prices.append((low + high) / 2)
    return prices


price_results = []
for i, row in sample.iterrows():
    prices = get_lookchem_prices_selenium(row["CAS"])
    avg_price = np.mean(prices) if prices else None
    price_results.append(avg_price)
    print(f"{i+1}/{len(sample)}: CAS {row['CAS']} -> 가격 {len(prices)}건, 평균 {avg_price}")
    time.sleep(1)  # 요청 사이 딜레이

driver.quit()

sample["price_usd"] = price_results
sample["has_price"] = sample["price_usd"].notna()
print(f"\n가격 확보: {sample['has_price'].sum()} / {len(sample)}")

1/20: CAS 51-21-8 -> 가격 19건, 평균 2837.1315789473683
2/20: CAS 4981-66-2 -> 가격 5건, 평균 242.1
3/20: CAS 92-88-6 -> 가격 14건, 평균 155.58214285714286
4/20: CAS 629-97-0 -> 가격 12건, 평균 113.58333333333333
5/20: CAS 115-76-4 -> 가격 6건, 평균 36.1
6/20: CAS 141-75-3 -> 가격 10건, 평균 37.81
7/20: CAS nan -> 가격 0건, 평균 None
8/20: CAS 57837-19-1 -> 가격 15건, 평균 140.7
9/20: CAS 3764-01-0 -> 가격 13건, 평균 246.67307692307693
10/20: CAS 120-75-2 -> 가격 15건, 평균 342.23333333333335
11/20: CAS 6190-65-4 -> 가격 3건, 평균 5.633333333333333
12/20: CAS 136-25-4 -> 가격 1건, 평균 2.4
13/20: CAS 105-34-0 -> 가격 14건, 평균 175.5
14/20: CAS 75-27-4 -> 가격 7건, 평균 28.235714285714288
15/20: CAS 85-19-8 -> 가격 11건, 평균 30.91818181818182
16/20: CAS 1871-96-1 -> 가격 7건, 평균 45.16428571428571
17/20: CAS 2197-37-7 -> 가격 0건, 평균 None
18/20: CAS 589-93-5 -> 가격 3건, 평균 39.36666666666667
19/20: CAS 580-22-3 -> 가격 9건, 평균 433.93333333333334
20/20: CAS 134-96-3 -> 가격 15건, 평균 205.48666666666668

가격 확보: 18 / 20


In [10]:
# ---------- 3. 물성/유해성 데이터셋 통합 (InChIKey 기준 중복 제거) ----------
mp_full = pd.read_csv("./data/processed/crc_mp_c.csv")[["SMILES", "InChIKey"]]
bp_full = pd.read_csv("./data/processed/crc_bp_c.csv")[["SMILES", "InChIKey"]]
tox21_full = pd.read_csv("./data/hazard/moleculenet_tox21.csv")[["SMILES", "InChIKey"]]

mp_full["in_mp"] = True
bp_full["in_bp"] = True
tox21_full["in_tox21"] = True

merged = mp_full.merge(bp_full, on=["SMILES", "InChIKey"], how="outer") \
                .merge(tox21_full, on=["SMILES", "InChIKey"], how="outer")

for col in ["in_mp", "in_bp", "in_tox21"]:
    merged[col] = merged[col].fillna(False)

merged = merged.drop_duplicates(subset="InChIKey").reset_index(drop=True)
print("통합 후 고유 화합물 수:", len(merged))
merged.head()

통합 후 고유 화합물 수: 14008


,SMILES,InChIKey,in_mp,in_bp,in_tox21
0,B[N+](C)(C)C,BNMCRXUHAWPWPT-UHFFFAOYSA-N,True,True,False
1,Br.COc1ccc2c(c1)C13CCCCC1C(C2)N(C)CC3,MISZALMBODQYFT-UHFFFAOYSA-N,True,False,False
2,Br.NCCBr,WJAXXWSZNSFVNG-UHFFFAOYSA-N,True,False,False
3,Br.NCCCBr,PQIYSSSTRHVOBW-UHFFFAOYSA-N,True,False,False
4,Br.Nc1ccccc1,KBPWECBBZZNAIE-UHFFFAOYSA-N,True,False,False


In [11]:
# ---------- 4. 전체 합집합에 CAS 번호 부여 (중간에 끊겨도 재실행 시 이어서 진행됨) ----------
import os

CAS_OUTPUT_PATH = "./data/processed/all_compounds_with_cas.csv"

if os.path.exists(CAS_OUTPUT_PATH):
    cas_df = pd.read_csv(CAS_OUTPUT_PATH)
    done_keys = set(cas_df["InChIKey"])
    print(f"기존 진행분 로드: {len(cas_df)}개 완료")
else:
    cas_df = pd.DataFrame(columns=list(merged.columns) + ["CAS"])
    done_keys = set()

todo = merged[~merged["InChIKey"].isin(done_keys)].reset_index(drop=True)
print(f"남은 작업: {len(todo)}개")

new_rows = []
for i, row in todo.iterrows():
    cas = get_cas_from_inchikey(row["InChIKey"])
    new_row = row.to_dict()
    new_row["CAS"] = cas
    new_rows.append(new_row)
    if (i + 1) % 50 == 0:
        cas_df = pd.concat([cas_df, pd.DataFrame(new_rows)], ignore_index=True)
        cas_df.to_csv(CAS_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"  {i+1}/{len(todo)} 진행, 중간 저장 완료")
    time.sleep(0.3)

if new_rows:
    cas_df = pd.concat([cas_df, pd.DataFrame(new_rows)], ignore_index=True)
    cas_df.to_csv(CAS_OUTPUT_PATH, index=False)

print(f"\nCAS 조회 완료: 전체 {len(cas_df)}개")

In [12]:
# ---------- 5. CAS 번호 없는 화합물 제거 ----------
cas_df_clean = cas_df.dropna(subset=["CAS"]).reset_index(drop=True)
print(f"CAS 있는 화합물: {len(cas_df_clean)} / {len(cas_df)}")
cas_df_clean.to_csv("./data/processed/all_compounds_with_cas.csv", index=False)
cas_df_clean.head()

CAS 있는 화합물: 12630 / 14008


,SMILES,InChIKey,in_mp,in_bp,in_tox21,CAS
0,Br.NCCCBr,PQIYSSSTRHVOBW-UHFFFAOYSA-N,True,False,False,5003-71-4
1,Br.Nc1ccccc1,KBPWECBBZZNAIE-UHFFFAOYSA-N,True,False,False,542-11-0
2,BrC(Br)(Br)Br,HJUGFYREWKUQJT-UHFFFAOYSA-N,True,True,False,558-13-4
3,BrC(Br)=C(Br)Br,OVRRJBSHBOXFQE-UHFFFAOYSA-N,True,True,False,79-28-7
4,BrC(Br)Br,DIKBFYAXUHHXCS-UHFFFAOYSA-N,True,True,True,75-25-2


In [14]:
# ---------- 6. Selenium 가격 크롤링 (원하는 만큼 배치로 나눠서 실행) ----------
BATCH_START = 0
BATCH_SIZE = 500
DELAY = 1.0

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import random

PRICE_OUTPUT_PATH = "./data/processed/all_compounds_with_price.csv"

cas_df_clean = pd.read_csv("./data/processed/all_compounds_with_cas.csv").dropna(subset=["CAS"]).reset_index(drop=True)

if os.path.exists(PRICE_OUTPUT_PATH):
    price_df = pd.read_csv(PRICE_OUTPUT_PATH)
    done_keys = set(price_df["InChIKey"])
    print(f"기존 진행분 로드: {len(price_df)}개 완료")
else:
    price_df = pd.DataFrame(columns=list(cas_df_clean.columns) + ["price_usd", "has_price"])
    done_keys = set()

remaining = cas_df_clean[~cas_df_clean["InChIKey"].isin(done_keys)].reset_index(drop=True)
batch = remaining.iloc[BATCH_START:BATCH_START + BATCH_SIZE].reset_index(drop=True)
print(f"이번 배치: {len(batch)}개 (전체 남은 {len(remaining)}개 중)")


def get_lookchem_prices_selenium(cas, driver):
    if pd.isna(cas):
        return []
    folder = cas.split("-")[0]
    url = f"https://www.lookchem.com/cas-{folder}/{cas}.html"
    try:
        driver.get(url)
        time.sleep(2)
    except Exception as e:
        print("  요청 에러:", e)
        return []

    page_source = driver.page_source
    if "FOB Price" not in page_source:
        return []

    prices = []
    for m in re.finditer(r"FOB Price:.*?USD\s*\$\s*([\d.]+)-([\d.]+)", page_source):
        low, high = float(m.group(1)), float(m.group(2))
        if low > 0 or high > 0:
            prices.append((low + high) / 2)
    return prices


options = Options()
driver = webdriver.Chrome(options=options)

new_rows = []
for i, row in batch.iterrows():
    prices = get_lookchem_prices_selenium(row["CAS"], driver)
    avg_price = np.mean(prices) if prices else None
    new_row = row.to_dict()
    new_row["price_usd"] = avg_price
    new_row["has_price"] = avg_price is not None
    new_rows.append(new_row)
    print(f"{i+1}/{len(batch)}: CAS {row['CAS']} -> 가격 {len(prices)}건, 평균 {avg_price}")

    if (i + 1) % 50 == 0:
        price_df = pd.concat([price_df, pd.DataFrame(new_rows)], ignore_index=True)
        price_df.to_csv(PRICE_OUTPUT_PATH, index=False)
        new_rows = []
        print(f"  중간 저장 완료 ({i+1}/{len(batch)})")

    time.sleep(DELAY)

driver.quit()

if new_rows:
    price_df.to_csv(PRICE_OUTPUT_PATH, index=False)

print(f"\n누적 가격 확보: {price_df['has_price'].sum()} / {len(price_df)}")

기존 진행분 로드: 50개 완료
이번 배치: 500개 (전체 남은 12580개 중)
1/500: CAS 626-15-3 -> 가격 0건, 평균 None
2/500: CAS 51632-16-7 -> 가격 0건, 평균 None
3/500: CAS 3163-27-7 -> 가격 0건, 평균 None
4/500: CAS 100-39-0 -> 가격 0건, 평균 None
5/500: CAS 3433-80-5 -> 가격 0건, 평균 None
